# Data Mining and Preparation using an API
Handling sensitive data, ethically. 

https://newsapi.org

The NewsAPI free tier only allows 100 requests per day. By default it returns the most recent articles, excluding articles from the past 24 hours, but will go back in time exactly one month if the request maximum allows. It doesn't directly return a location. This means that for us to connect the retrieved data with our other datasets, we will need to extract the location from the returned text. Because of the limitations on the free API (the paid tiers cost 449 and 1749 dollars per month), we can use the returned data to create a snapshot in time, but it is by no means a comprehensive approach to news article analysis. Because of these limitations, I will be pivoting the inclusion of the API by creating a boolean column when I merge the datasets for each row, indicating whether or not the state has been included in the news api's most recent and relevant 100 publications that reported on school shootings and closely related topics. I will also be including an average sentiment analysis score.

## Code Attribution

<ul>
  <li> Title: The Data Wrangling Workshop
  <li> Author: Brian Lipp, Shubhadeep Roychowdhury, Dr. Tirthajyoti Sarkar
  <li> Date: 16 November 2025
  <li> Modified By: Charlotte Alley
  <li> Description: Advanced web scraping and data gathering

<ul>
  <li> Title: Sentiment Analysis using VADER
  <li> Author: Geeksforgeeks.org
  <li> Date: 16 November 2025
  <li> Modified By: Charlotte Alley
  <li> Description: It uses a pre-built lexicon of words associated with sentiment values and applies specific rules to calculate sentiment scores. 

In [1]:
import pandas as pd
import requests

In [2]:
news_url = "https://newsapi.org/v2/everything?"
q = "+school +shooting"
apiKey = "c2b6ea9818094cfc957fc95c141cdaf6"
sortBy = "relevancy"

# Uses defaults for other parameters, in order to stay within 100 retreival 
# per day limitation, defaults automatically go back one month
params = {
    "q": q, 
    "apiKey": apiKey, 
    "sortBy": sortBy
}

# Establishes article list to append retrieved article data to
articles_list = []

# Makes get request with specified variables and parameters 
response = requests.get(news_url, params=params)
# Catches statust code errors, only continues with 'okay' response of 200
if response.status_code != 200: 
    print("Failed to retrieve data:", response.json().get("message"))
else: 
    # Assigns json response data to variable 
    data = response.json()
    # Uses another get request on data variable, further parsing response
    articles = data.get("articles", [])
    if not articles:
        print("No articles found.")
    else:
        for article in articles:
            articles_list.append({
                "title": article.get("title", ""),
                "description": article.get("description", ""),
                "content": article.get("content", "")
            })

# Creates dataframe from list
news_df = pd.DataFrame(articles_list)

In [3]:
# Prints first five rows of new dataframe 
news_df.head()

,title,description,content
0,Middle school locked down on reports of possib...,None,"If you click 'Accept all', we and our partners..."
1,A former teacher shot her 6-year-old student w...,Abby Zwerner's lawsuit that accused an adminis...,"NEWPORT NEWS, Va. A jury in Virginia on Thursd..."
2,Meet the Palestinian Teens Trying to Win Robot...,"Next week, five teens from Palestine will head...",Both also serve as reminders to current team m...
3,Shooting leaves 1 dead and 6 hurt at historica...,One person was killed and six more were wounde...,"LINCOLN UNIVERSITY, Pa. One person was killed ..."
4,Jury reaches verdict in Richneck Elementary sh...,"The jury ruled in favor of the plaintiff, Abig...","NEWPORT NEWS, Va. (WAVY) The jury reached a ve..."


## Step 1: Extract State 
I tried a few different options to extract the state information (including the spaCy library), but using an incredibly specific regular expression was the only method that didn't return numerous false results. The parts of this pattern that led to its success included the capitalization, the proceeding space and comma requirement for the state abbreviations, and the word boundary for both patterns to avoid matching to any state abbreviation within longer words. In the end, we were still left with many null values, but that's because not every article had location information in its returned content. This makes sense, becuase many articles are written and published each day that have nothing to do with state-specific happenings. 

In [4]:
import re

# Creates list of uppercase state names for regex matching
full_state_names = [
    "ALABAMA", "ALASKA", "ARIZONA", "ARKANSAS", "CALIFORNIA", "COLORADO",
    "CONNECTICUT", "DELAWARE", "FLORIDA", "GEORGIA", "HAWAII", "IDAHO",
    "ILLINOIS", "INDIANA", "IOWA", "KANSAS", "KENTUCKY", "LOUISIANA",
    "MAINE", "MARYLAND", "MASSACHUSETTS", "MICHIGAN", "MINNESOTA",
    "MISSISSIPPI", "MISSOURI", "MONTANA", "NEBRASKA", "NEVADA",
    "NEW HAMPSHIRE", "NEW JERSEY", "NEW MEXICO", "NEW YORK",
    "NORTH CAROLINA", "NORTH DAKOTA", "OHIO", "OKLAHOMA", "OREGON",
    "PENNSYLVANIA", "RHODE ISLAND", "SOUTH CAROLINA", "SOUTH DAKOTA",
    "TENNESSEE", "TEXAS", "UTAH", "VERMONT", "VIRGINIA", "WASHINGTON",
    "WEST VIRGINIA", "WISCONSIN", "WYOMING"
]

# Creates list of upppercase state abbreviations for regex matching
state_abbreviations = [
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN',
    'IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV',
    'NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN',
    'TX','UT','VT','VA','WA','WV','WI','WY'
]

# Defines regex pattern that finds full state names anywhere as whole words,
# uses word boundaries for exact matches 
full_state_pattern = r'\b(' + '|'.join(full_state_names) + r')\b'

# Defines regex pattern that finds state abbreviations preceded by comma and 
# space, ensures abbreviations matched as separate whole words
abbr_state_pattern = r',\s(' + '|'.join(state_abbreviations) + r')\b'

# Combines pattern using pipe symbol, signifying OR 
pattern = re.compile(full_state_pattern + '|' + abbr_state_pattern)

def extract_state(text):    
    """
    Extracts full or abbreviate state name from input. Converts text to 
    uppercase for case-insensitive matching. Returns matched state name or 
    abbreviation, or None if no match is found.
    """
    if pd.isnull(text):
        return None
    upper_text = text.upper()
    match = pattern.search(upper_text)
    # Returns first match only to avoid duplicate returns
    if match:
        return match.group(1) if match.group(1) else match.group(2)
    return None

# Creates new column with extracted state information
news_df['location'] = news_df['content'].apply(extract_state)

# Outputs unique value count
news_df['location'].value_counts()

location
VIRGINIA          9
TENNESSEE         6
VA                3
KANSAS            2
INDIANA           2
MISSISSIPPI       2
PA                1
MICHIGAN          1
ARIZONA           1
TEXAS             1
NEW YORK          1
MINNESOTA         1
OKLAHOMA          1
OREGON            1
SOUTH CAROLINA    1
FLORIDA           1
Name: count, dtype: int64

In [5]:
# Outputs non-null numbers for reference on how many states were extracted
news_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        96 non-null     object
 1   description  95 non-null     object
 2   content      96 non-null     object
 3   location     34 non-null     object
dtypes: object(4)
memory usage: 3.1+ KB


## Step 2: Standardize text
We are working with a lot of text, so my next step will be to standardize and preprocess all of the text in the dataframe for easier processing. 

In [6]:
import string  

def preprocess_text(text):
    # Checks if input is string
    if isinstance(text, str):
        # Converts all text to lowercase
        text = text.lower()
        # Normalizes spacing within text
        text = re.sub(r'\s+', ' ', text)
        # Removes all punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        # Outputs clean string without leading or trailing whitespaces
        return text.strip()
    else: 
        return text 

# Runs on all categorical columns 
news_df_cleaned = news_df.apply(lambda col: col.map(preprocess_text) if 
                                col.dtype == 'object' else col)

In [7]:
news_df_cleaned.head()

,title,description,content,location
0,middle school locked down on reports of possib...,None,if you click accept all we and our partners in...,None
1,a former teacher shot her 6yearold student win...,abby zwerners lawsuit that accused an administ...,newport news va a jury in virginia on thursday...,va
2,meet the palestinian teens trying to win robot...,next week five teens from palestine will head ...,both also serve as reminders to current team m...,None
3,shooting leaves 1 dead and 6 hurt at historica...,one person was killed and six more were wounde...,lincoln university pa one person was killed an...,pa
4,jury reaches verdict in richneck elementary sh...,the jury ruled in favor of the plaintiff abiga...,newport news va wavy the jury reached a verdic...,va


## Step 3: Use Fuzzy Matching
Even though we specified our query parameter to include the words school and shooting, it's clear by looking at the title values that we ended up with some articles that have nothing to do with school shootings or gun violence in general. I'm going to use fuzzy matching to create a dataframe with only the relevent articles. 

In [8]:
from thefuzz import fuzz

# Keywords created by looking through titles, Charlie Kirk and Luigi Mangione 
# included as key figures involved in recent gun violence incidents 
keywords = ['shooting', 'shot', 'dead', 'killed', 'arrested', 'gun', 'suspect', 
           'charlie kirk', 'sandy hook', 'firing', 'tragedy', 'lockdown', 
           'gunfire', 'massacre', 'luigi mangione', "assault", "homicide", 
            "firearms", "injured", "victim", "fatal", "school shooting", 
            "mass shooting"]

# Creates fuzzy matching function with 80 threshold
def fuzzy_match_any(text, keywords, threshold=80):
    return any(fuzz.token_set_ratio(text, kw) >= threshold for kw in keywords)

# Creates new dataframe with standardized and filtered data 
filtered_df = news_df_cleaned[news_df_cleaned['title']
    .apply(lambda x: fuzzy_match_any(x, keywords))]

In [9]:
# Shows reduction in row number for filtered data  
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50 entries, 0 to 93
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        50 non-null     object
 1   description  49 non-null     object
 2   content      50 non-null     object
 3   location     25 non-null     object
dtypes: object(4)
memory usage: 2.0+ KB


In [10]:
# Prints list of titles to verify that titles are relevent
filtered_df['title'].values

array(['middle school locked down on reports of possible shooting',
       'a former teacher shot her 6yearold student wins a 10 million jury verdict',
       'shooting leaves 1 dead and 6 hurt at historically black universitys homecoming fest in pennsylvania',
       'jury reaches verdict in richneck elementary shooting case',
       'loaded gun found in student’s backpack outside belton school suspect arrested',
       'trial starts in 40 million lawsuit filed by a teacher who was shot by a 6yearold student',
       'almont workplace shooting suspect charged victim identified as grand blanc football coach',
       'man formally charged after road rage escalates to shooting near multiple businesses',
       'jury awards 10 million to teacher who was shot by 6yearold student',
       'a former teacher shot by student 6 wins 10m jury verdict against exassistant principal',
       'suspect arrested in shooting of legendary oakland coach john beam at laney college police',
       'coroner

## Step 4: Handle Missing Values 
We are only interested in articles that have state-specific information, so I will be removing all rows that that ended up with a null value for location. This leaves us without any missing values.

In [11]:
# Filters out rows with null values in the location column
state_df = filtered_df[filtered_df['location'].notnull()]
state_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25 entries, 1 to 92
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        25 non-null     object
 1   description  25 non-null     object
 2   content      25 non-null     object
 3   location     25 non-null     object
dtypes: object(4)
memory usage: 1000.0+ bytes


## Step 5: Recode Location 
I am going to standardize the state names, so that they match our other two datasets. I want to make sure this step works for all states, since the API pull will change every day. 

In [12]:
# Creates dictionary with replacement values for column
mapping = {
    'al': 'Alabama', 'ak': 'Alaska', 'az': 'Arizona', 'ar': 'Arkansas',
    'ca': 'California', 'co': 'Colorado', 'ct': 'Connecticut', 'de': 'Delaware',
    'fl': 'Florida', 'ga': 'Georgia', 'hi': 'Hawaii', 'id': 'Idaho',
    'il': 'Illinois', 'in': 'Indiana', 'ia': 'Iowa', 'ks': 'Kansas',
    'ky': 'Kentucky', 'la': 'Louisiana', 'me': 'Maine', 'md': 'Maryland',
    'ma': 'Massachusetts', 'mi': 'Michigan', 'mn': 'Minnesota', 
    'ms': 'Mississippi', 'mo': 'Missouri', 'mt': 'Montana', 'ne': 'Nebraska', 
    'nv': 'Nevada', 'nh': 'New Hampshire', 'nj': 'New Jersey', 
    'nm': 'New Mexico', 'ny': 'New York', 'nc': 'North Carolina', 
    'nd': 'North Dakota', 'oh': 'Ohio', 'ok': 'Oklahoma', 'or': 'Oregon', 
    'pa': 'Pennsylvania', 'ri': 'Rhode Island', 'sc': 'South Carolina',
    'sd': 'South Dakota', 'tn': 'Tennessee', 'tx': 'Texas', 'ut': 'Utah',
    'vt': 'Vermont', 'va': 'Virginia', 'wa': 'Washington', 
    'wv': 'West Virginia', 'wi': 'Wisconsin', 'wy': 'Wyoming'
}

# Function handles spelled out states as well, capitalizing the first letter of 
# each word in the state 
def map_abbreviations(val, mapping):
    return mapping[val] if val in mapping else val.title()

# Creates copy to silence warning 
state_df = filtered_df[filtered_df['location'].notnull()].copy()

# Uses apply and map_abbreviations function to replace values in column
state_df['location'] = state_df['location'].apply(lambda x: map_abbreviations(
    x.lower(), mapping))

state_df.head()

,title,description,content,location
1,a former teacher shot her 6yearold student win...,abby zwerners lawsuit that accused an administ...,newport news va a jury in virginia on thursday...,Virginia
3,shooting leaves 1 dead and 6 hurt at historica...,one person was killed and six more were wounde...,lincoln university pa one person was killed an...,Pennsylvania
4,jury reaches verdict in richneck elementary sh...,the jury ruled in favor of the plaintiff abiga...,newport news va wavy the jury reached a verdic...,Virginia
5,loaded gun found in student’s backpack outside...,student arrested and charged after bringing lo...,kansas city mo a quickthinking tip from a stud...,Kansas
6,trial starts in 40 million lawsuit filed by a ...,a former assistant principal at a virginia ele...,a former assistant principal at a virginia ele...,Virginia


In [13]:
state_df['location'].unique()

array(['Virginia', 'Pennsylvania', 'Kansas', 'Michigan', 'Mississippi',
       'Arizona', 'Tennessee', 'Minnesota', 'New York', 'Oklahoma',
       'Indiana', 'South Carolina', 'Florida'], dtype=object)

## Step 6: Conduct Sentiment Analysis 
I want to create a new column with sentiment analysis to gain a little more insight into how sentiment analysis averages compare across states reported on. 

In [14]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Initializes VADER sentiment analyzer 
analyzer = SentimentIntensityAnalyzer()

# Defines function to classify sentiment using VADER compound score, 1 if 
# positive, 0 if negative
def classify_sentiment_vader(text):
    if not isinstance(text, str) or not text.strip():
        return None  
    scores = analyzer.polarity_scores(text)
    compound = scores['compound']
    return 1 if compound > -0.05 else 0

# Applies VADER to each relevant column, creates new columns with 
# classifications to later average
state_df['title_sentiment'] = state_df['title'].apply(classify_sentiment_vader)
state_df['description_sentiment'] = state_df['description'].apply(
    classify_sentiment_vader)
state_df['content_sentiment'] = state_df['content'].apply(
    classify_sentiment_vader)

# Averages sentiment score columns
state_df['avg_sentiment'] = state_df[
    ['title_sentiment', 'description_sentiment', 'content_sentiment']
].mean(axis=1).round(2)

In [15]:
state_df.head()

,title,description,content,location,title_sentiment,description_sentiment,content_sentiment,avg_sentiment
1,a former teacher shot her 6yearold student win...,abby zwerners lawsuit that accused an administ...,newport news va a jury in virginia on thursday...,Virginia,1,0,0,0.33
3,shooting leaves 1 dead and 6 hurt at historica...,one person was killed and six more were wounde...,lincoln university pa one person was killed an...,Pennsylvania,0,0,0,0.00
4,jury reaches verdict in richneck elementary sh...,the jury ruled in favor of the plaintiff abiga...,newport news va wavy the jury reached a verdic...,Virginia,1,0,1,0.67
5,loaded gun found in student’s backpack outside...,student arrested and charged after bringing lo...,kansas city mo a quickthinking tip from a stud...,Kansas,0,0,0,0.00
6,trial starts in 40 million lawsuit filed by a ...,a former assistant principal at a virginia ele...,a former assistant principal at a virginia ele...,Virginia,0,0,0,0.00


In [16]:
# Displays descriptive statistics for sentiment scores 
state_df.describe()

,title_sentiment,description_sentiment,content_sentiment,avg_sentiment
count,25.000000,25.000000,25.000000,25.000000
mean,0.480000,0.360000,0.280000,0.374000
std,0.509902,0.489898,0.458258,0.377547
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.330000
75%,1.000000,1.000000,1.000000,0.670000
max,1.000000,1.000000,1.000000,1.000000


## Step 7: Remove Unnecessary Columns, Reorder and Rename Columns
I could have kept the individual sentiment scores, but since we will be merging on state, it makes more sense to have one score per state. For that reason, I will be removing the unnecessary columns, putting the state column first, renaming the columns, and averaging any sentiment scores that share a state value. Since not every state is represented in the most relevant and most recent articles in our search query, the merge will create missing values. Missing states will be represented as "not included in api results". 

In [17]:
# Reorders columns
final_df = state_df[['location', 'avg_sentiment']]

# Outputs first five rows
final_df.head()

,location,avg_sentiment
1,Virginia,0.33
3,Pennsylvania,0.00
4,Virginia,0.67
5,Kansas,0.00
6,Virginia,0.00


In [19]:
# Renames columns 
final_df.columns = ['State', 'Article Sentiment Average']

# Calculates mean for included states, each state is only represented once
grouped_final_df = final_df.groupby('State', as_index=False).mean()

# Human readable dataset
grouped_final_df.head()

# Save to csv file for database
grouped_final_df.to_csv('news_api.csv', index=False)

## Ethical Implications:
The changes made to the dataset include: extracting the state from the 'content' column, standardizing the text, using fuzzy matching to re-filter relevant titles, handling missing values, recoding the location values, conducting sentiment analysis, removing unnecessary columns, and reording and renaming columns. 

There are legal and regulatory guidelines for the data. The api website has a terms of service section that states: 

<li>Unless applicable by law or expressly permitted by the owner, you will not, and will not permit your end users or others acting on your behalf to, do any of the following with the Data returned from the API:
<li>-Misrepresent the ownership or the source;
<li>-Obscure, remove or change any copyright, trademark, or other proprietary notices;
<li>-Falsify or delete any author attributions, legal notices, or other labels of the origin or source of material.
<li>-Unless expressly authorized herein, you may not publicly release or disclose any data or usage statistics or other information regarding the Data.</li>

There are many risks that could be created with the transformations completed. While the regex method was the most accurate extraction tool, states that do not follow this pattern (for example, a misspelled state) will not be caught in the filter. The method will also catch the first state mentioned, even if this state is not where the shooting or related event occured. This risks introducing error and bias to the dataset, as the sentiment score will be effected by state article inclusions and ommissions. Standardizing text can introduce errors by interfering with the sentiment analysis (for example, removing a dash from between two words will affect their meaning). Fuzzy matching requires the tuning of a threshold, creating risk for both false negatives and false positives, potentially omitting or including articles that don't align with our analysis. Removing missing values can also lead to bias, but in this case, it didn't make sense to include articles that weren't state-specific. Recoding the location values has the potential to introduce errors into the dataset, and conducting sentiment analysis has the potential to create information loss, as a numerous stringed article is replaced with a single number. Removing unnecessary columns also risks losing information and context, as the data has been entirely transformed into two new variables. 

In cleaning and transforming the data, I made the assumption that the values on the dataset were accurate, and relevant to the problem that we're exploring. The data has been compiled from over 150,000 worldwide sources, each at risk for their own editorial bias or factual inaccuracies. Since I don't have a way to validate the VADER sentiment analysis other than human validation, I've made the assumption that the assessments are accurate. I've also made the assumption that the most relevent and recent 100 articles matching the school shooting query are a representative sample for visualization purposes. 

The api returns current and historic news articles published by over 150,000 worldwide sources. URL's are provided if requested, and private details are included as published. The api claims to be used by over 500,000 developers (including Walmart), and includes testimonials. It sources its data through a combination of web scraping and news partnerships.  

Moving forward with the ethical implications that we have identified, clearly documenting assumptions and transformations for transparency will be key. With the api 100 requests limitations, the dataset is incomplete, which needs to be communicated in the visualizations it goes on to form. The sentiment analysis limitations also need to be communicated, and any conclusions drawn from the visualizations must be seen as preliminary, and a basis from which to do further research.